# Módulo F2 · SQL para análisis de datos — Práctica guiada

**Caso:** eres analista en *TiendaSur*. Dirección quiere responder estas preguntas con datos:

1. ¿Qué categorías venden más?
2. ¿Qué productos son los más rentables por categoría?
3. ¿Cómo evolucionan las ventas mes a mes y cuál es el **acumulado**?

Todos los datos son sintéticos y se crean en la celda siguiente. La base de datos es **SQLite** (un archivo local) y la consultamos con `pandas.read_sql`.

> **Ejecuta las celdas en orden** (el kernel no recuerda entre sesiones).


In [ ]:
import sqlite3
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# --- Base de datos sintética de TiendaSur -------------------------------------
clientes = pd.DataFrame({
    "cliente_id": range(1, 41),
    "nombre": [f"Cliente {i:02d}" for i in range(1, 41)],
    "ciudad": rng.choice(["Madrid", "Barcelona", "Valencia", "Sevilla"], 40, p=[.35, .3, .2, .15]),
})

categorias = ["Bebidas", "Alimentación", "Limpieza", "Electrónica"]
productos = pd.DataFrame({
    "producto_id": range(1, 25),
    "nombre": [f"Producto {i:02d}" for i in range(1, 25)],
    "categoria": np.tile(categorias, 6),
    "precio": np.round(rng.uniform(1, 60, 24), 2),
})
productos.loc[3, "precio"] = np.nan   # un nulo para practicar COALESCE

# Pedidos: 400 pedidos repartidos en 2024
n_pedidos = 400
pedidos = pd.DataFrame({
    "pedido_id": range(1, n_pedidos + 1),
    "cliente_id": rng.integers(1, 41, n_pedidos),
    "fecha": pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 364, n_pedidos), unit="D"),
})

# Líneas de pedido: entre 1 y 4 productos por pedido
lineas = []
for _, p in pedidos.iterrows():
    n_l = rng.integers(1, 5)
    elegidos = rng.choice(productos["producto_id"], size=min(n_l, 24), replace=False)
    for pid in elegidos:
        unid = rng.integers(1, 8)
        precio = float(productos.loc[productos.producto_id == pid, "precio"].iloc[0])
        if np.isnan(precio):
            precio = 10.0   # el ticket nunca es nulo: precio medio como fallback
        lineas.append({"pedido_id": int(p["pedido_id"]), "producto_id": int(pid),
                       "unidades": int(unid), "precio_unitario": precio})
lineas = pd.DataFrame(lineas)
lineas["importe"] = lineas["unidades"] * lineas["precio_unitario"]

# Guardamos en SQLite
conn = sqlite3.connect(":memory:")
for nombre, df in [("clientes", clientes), ("productos", productos), ("pedidos", pedidos), ("lineas", lineas)]:
    df.to_sql(nombre, conn, index=False)

print(f"clientes: {len(clientes)}  productos: {len(productos)}  pedidos: {len(pedidos)}  líneas: {len(lineas)}")
conn

# Soluciones del módulo F2

## E1 · Ingresos y pedidos por ciudad

In [ ]:
pd.read_sql("""
SELECT c.ciudad,
       COUNT(DISTINCT p.pedido_id) AS pedidos,
       ROUND(SUM(l.importe), 2) AS ingresos
FROM clientes c
JOIN pedidos p ON p.cliente_id = c.cliente_id
JOIN lineas l ON l.pedido_id = p.pedido_id
GROUP BY c.ciudad
ORDER BY ingresos DESC;
""", conn)

## E2 · Top 5 clientes por gasto

In [ ]:
pd.read_sql("""
SELECT c.nombre, ROUND(SUM(l.importe), 2) AS gasto
FROM clientes c
JOIN pedidos p ON p.cliente_id = c.cliente_id
JOIN lineas l ON l.pedido_id = p.pedido_id
GROUP BY c.cliente_id
ORDER BY gasto DESC
LIMIT 5;
""", conn)

## E3 · Ticket medio por mes (versión CTE)

In [ ]:
pd.read_sql("""
WITH por_pedido AS (
  SELECT strftime('%Y-%m', p.fecha) AS mes,
         p.pedido_id,
         SUM(l.importe) AS importe_pedido
  FROM lineas l
  JOIN pedidos p ON p.pedido_id = l.pedido_id
  GROUP BY mes, p.pedido_id
)
SELECT mes,
       ROUND(SUM(importe_pedido), 2) AS ingresos,
       COUNT(*) AS pedidos,
       ROUND(AVG(importe_pedido), 2) AS ticket_medio
FROM por_pedido
GROUP BY mes
ORDER BY mes;
""", conn)

## E4 · Cuota dentro de la categoría

In [ ]:
pd.read_sql("""
WITH por_producto AS (
  SELECT pr.categoria, pr.nombre, SUM(l.unidades) AS unidades
  FROM lineas l
  JOIN productos pr ON pr.producto_id = l.producto_id
  GROUP BY pr.producto_id
)
SELECT categoria, nombre, unidades,
       ROUND(100.0 * unidades / SUM(unidades) OVER (PARTITION BY categoria), 2) AS cuota_pct
FROM por_producto
ORDER BY categoria, cuota_pct DESC;
""", conn)

## E5 · Pedidos con más líneas que el pedido 100

In [ ]:
pd.read_sql("""
WITH lineas_pedido AS (
  SELECT pedido_id, COUNT(*) AS n_lineas
  FROM lineas
  GROUP BY pedido_id
),
ref AS (
  SELECT n_lineas FROM lineas_pedido WHERE pedido_id = 100
)
SELECT lp.pedido_id, lp.n_lineas
FROM lineas_pedido lp, ref
WHERE lp.n_lineas > ref.n_lineas
ORDER BY lp.n_lineas DESC;
""", conn)